# Exercice 3.4 — Dask pour datasets volumineux

Objectif du notebook:
- charger les 12 fichiers avec `dask.dataframe.read_parquet("data/raw/*.parquet")`,
- reproduire les calculs de l'exercice 3.2 (`groupby` + `pivoting`),
- comparer Pandas et Dask sur le temps d'exécution,
- observer le pic mémoire côté Dask via le dashboard,
- vérifier l'égalité des résultats après `.compute()`.

Ce notebook sépare volontairement:
1. un benchmark **équitable** sur un sous-ensemble de fichiers,
2. un benchmark **Dask full 12 mois**,
3. un benchmark **Pandas full** désactivé par défaut pour éviter un OOM.

In [ ]:
from __future__ import annotations

import json
import sys
import tracemalloc
from pathlib import Path
from time import perf_counter

import dask.dataframe as dd
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
from dask.distributed import Client
from IPython.display import display
from pandas.testing import assert_frame_equal

PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / "src").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

sys.path.insert(0, str(PROJECT_ROOT / "src"))

from features.encoding import (
    build_activity_dashboard,
    build_activity_dashboard_dask,
    load_raw_with_dask,
    pivot_activity_table,
)

sns.set_theme(style="whitegrid")
pd.options.display.max_columns = 40
pd.options.display.float_format = "{:,.4f}".format

## Paramètres

- `SUBSET_FILE_COUNT` sert au benchmark équitable Pandas vs Dask.
- `RUN_FULL_DASK` exécute Dask sur les 12 mois.
- `RUN_FULL_PANDAS` reste `False` par défaut pour éviter de tuer le kernel.

In [ ]:
DATA_DIR = PROJECT_ROOT / "data" / "raw"
DATA_PATTERN = str(DATA_DIR / "yellow_tripdata_2023-*.parquet")
PARQUET_FILES = sorted(DATA_DIR.glob("yellow_tripdata_2023-*.parquet"))

COLUMNS = [
    "PULocationID",
    "tpep_pickup_datetime",
    "total_amount",
    "tip_amount",
    "fare_amount",
]

SUBSET_FILE_COUNT = 3
RUN_FULL_DASK = True
RUN_FULL_PANDAS = False
START_DASK_CLIENT = True

subset_files = PARQUET_FILES[:SUBSET_FILE_COUNT]
print("Project root:", PROJECT_ROOT)
print("Files found:", len(PARQUET_FILES))
print("Subset files:", [path.name for path in subset_files])

In [ ]:
client = None
if START_DASK_CLIENT:
    client = Client(processes=True)
    print(client)
    print("Dask dashboard:", client.dashboard_link)

## Helpers

On recrée les colonnes nécessaires à l'exercice 3.2, notamment `tip_pct`, avant de lancer les agrégations.

In [ ]:
def prepare_pandas_frame(paths: list[Path]) -> pd.DataFrame:
    frames = []
    for path in paths:
        df = pd.read_parquet(path, columns=COLUMNS)
        df = df.loc[df["fare_amount"] > 0].copy()
        df["tip_pct"] = df["tip_amount"] / df["fare_amount"]
        frames.append(df)
    return pd.concat(frames, ignore_index=True)


def prepare_dask_frame(pattern_or_paths: str | list[Path]):
    if isinstance(pattern_or_paths, list):
        ddf = dd.read_parquet([str(path) for path in pattern_or_paths], columns=COLUMNS)
    else:
        ddf = load_raw_with_dask(pattern_or_paths, columns=COLUMNS)
    ddf = ddf[ddf["fare_amount"] > 0]
    ddf = ddf.assign(tip_pct=ddf["tip_amount"] / ddf["fare_amount"])
    return ddf


def benchmark(label: str, func, *args, **kwargs):
    tracemalloc.start()
    started_at = perf_counter()
    result = func(*args, **kwargs)
    elapsed_s = perf_counter() - started_at
    _, peak_bytes = tracemalloc.get_traced_memory()
    tracemalloc.stop()
    return {
        "label": label,
        "elapsed_s": elapsed_s,
        "peak_python_mb": peak_bytes / (1024 ** 2),
        "result": result,
    }


def compute_activity_and_pivot_pandas(paths: list[Path]):
    df = prepare_pandas_frame(paths)
    activity = build_activity_dashboard(df).sort_index()
    pivot = pivot_activity_table(activity).sort_index()
    return activity, pivot


def compute_activity_and_pivot_dask(pattern_or_paths: str | list[Path]):
    ddf = prepare_dask_frame(pattern_or_paths)
    activity = build_activity_dashboard_dask(ddf).compute().sort_index()
    pivot = pivot_activity_table(activity).sort_index()
    return activity, pivot


def summarize_activity(activity: pd.DataFrame) -> dict:
    return {
        "rows": int(len(activity)),
        "trip_count_sum": float(activity["trip_count"].sum()),
        "revenue_total_sum": float(activity["revenue_total"].sum()),
        "tip_pct_mean_mean": float(activity["tip_pct_mean"].mean()),
    }

## Benchmark équitable Pandas vs Dask sur le même sous-ensemble

C'est la comparaison la plus rigoureuse pour le rapport: même input, mêmes calculs, mêmes résultats attendus.

In [ ]:
pandas_bench = benchmark("pandas_subset", compute_activity_and_pivot_pandas, subset_files)
dask_bench = benchmark("dask_subset", compute_activity_and_pivot_dask, subset_files)

pandas_activity, pandas_pivot = pandas_bench["result"]
dask_activity, dask_pivot = dask_bench["result"]

assert_frame_equal(pandas_activity, dask_activity)
assert_frame_equal(pandas_pivot, dask_pivot)

print("Equality check activity: OK")
print("Equality check pivot: OK")
print(json.dumps(summarize_activity(pandas_activity), indent=2))

In [ ]:
subset_benchmark_df = pd.DataFrame([
    {k: v for k, v in pandas_bench.items() if k != "result"},
    {k: v for k, v in dask_bench.items() if k != "result"},
])
subset_benchmark_df

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

sns.barplot(data=subset_benchmark_df, x="label", y="elapsed_s", ax=axes[0], palette="Blues_d")
axes[0].set_title("Temps d'exécution")
axes[0].set_ylabel("secondes")
axes[0].set_xlabel("")

sns.barplot(data=subset_benchmark_df, x="label", y="peak_python_mb", ax=axes[1], palette="Greens_d")
axes[1].set_title("Pic mémoire Python (tracemalloc)")
axes[1].set_ylabel("MB")
axes[1].set_xlabel("")

plt.tight_layout()

## Résultats intermédiaires

Tu peux afficher ici les DataFrames finaux pour vérifier visuellement la cohérence des agrégations.

In [ ]:
display(pandas_activity.head())
display(pandas_pivot.head())

## Dask sur les 12 mois

Cette cellule répond directement à l'exigence 61 du sujet (`read_parquet("data/raw/*.parquet")`).

Pour le pic mémoire réel côté Dask, observe le dashboard affiché dans `client.dashboard_link`, généralement `http://localhost:8787`.

In [ ]:
if RUN_FULL_DASK:
    full_dask_bench = benchmark("dask_full_12_months", compute_activity_and_pivot_dask, DATA_PATTERN)
    full_dask_activity, full_dask_pivot = full_dask_bench["result"]
    display(pd.DataFrame([{k: v for k, v in full_dask_bench.items() if k != "result"}]))
    print(json.dumps(summarize_activity(full_dask_activity), indent=2))
else:
    print("RUN_FULL_DASK=False -> cellule non exécutée.")

## Pandas sur les 12 mois

Désactivé par défaut. Sur une machine limitée, cette cellule peut provoquer un OOM.

In [ ]:
if RUN_FULL_PANDAS:
    full_pandas_bench = benchmark("pandas_full_12_months", compute_activity_and_pivot_pandas, PARQUET_FILES)
    full_pandas_activity, full_pandas_pivot = full_pandas_bench["result"]
    display(pd.DataFrame([{k: v for k, v in full_pandas_bench.items() if k != "result"}]))
    print(json.dumps(summarize_activity(full_pandas_activity), indent=2))
else:
    print("RUN_FULL_PANDAS=False -> non exécuté pour protéger la mémoire.")

## Réponse à mettre dans le rapport

Question 63 — interprétation attendue:
- **Dask devient intéressant** quand le dataset ne tient plus confortablement en mémoire Pandas ou quand plusieurs partitions peuvent être exploitées en parallèle.
- **L'overhead du graphe de calcul l'emporte** sur les petits volumes, où Pandas reste souvent plus rapide grâce à un coût fixe plus faible.
- La comparaison rigoureuse se fait sur le **subset identique** validé par `assert_frame_equal(...)`.
- La comparaison opérationnelle sur les **12 mois complets** se fait surtout via Dask, avec observation du dashboard pour le pic mémoire réel.